# Notebook 07 — Visualizaciones con matplotlib 📊

Hasta ahora exploraste los datos con **números** (shape, conteos, promedios). Hoy aprendes algo igual de importante: **explorarlos con gráficos**.

Una imagen bien hecha dice más en 1 segundo que una tabla de 200 filas. Y a la inversa: hay patrones (sesgos, outliers, relaciones) que **solo se ven con un gráfico**.

Vamos a usar **matplotlib**, la librería de visualización más popular de Python. Es la base de casi todas las demás (seaborn, pandas plot, etc.).

## Objetivos de aprendizaje

1. Entender el patrón **`fig, ax = plt.subplots()`** y por qué lo usamos.
2. Hacer un **histograma** para ver la distribución de una variable numérica.
3. Hacer un **gráfico de barras** para comparar valores por categoría.
4. Hacer un **scatter plot** para ver la relación entre dos variables numéricas.
5. Hacer un **boxplot** para comparar distribuciones por categoría.
6. **Personalizar** título, etiquetas y tamaño del gráfico.

---

## 1. Setup

Cargamos `titanic` y lo limpiamos como en el Notebook 06.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Load and clean (same recipe as NB06)
df = sns.load_dataset("titanic")
df_clean = df.drop(columns=["deck"]).copy()
df_clean["age"] = df_clean["age"].fillna(df_clean["age"].median())
df_clean = df_clean.dropna(subset=["embarked"]).reset_index(drop=True)

print(f"df_clean: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns")
df_clean.head(3)

---

## 2. El patrón `fig, ax = plt.subplots()`

Casi todos los gráficos de matplotlib se construyen con esta línea:

```python
fig, ax = plt.subplots(figsize=(8, 5))
```

- **`fig`** (*figure*) — el lienzo completo (puede contener varios gráficos).
- **`ax`** (*axes*) — el gráfico individual donde dibujas (uno solo, en este notebook).

Después llamas métodos sobre `ax` para añadir contenido:

```python
ax.hist(...)              # añade un histograma
ax.bar(...)               # añade barras
ax.scatter(...)           # añade puntos
ax.set_title("...")       # pone título
ax.set_xlabel("...")      # etiqueta eje X
ax.set_ylabel("...")      # etiqueta eje Y
plt.show()                # muestra el gráfico
```

> 💡 Hay una forma "rápida" (`plt.hist(...)`) que escribe directamente sobre el último `ax` activo. Pero **el patrón `fig, ax`** es más explícito, más fácil de leer cuando el código crece, y el estándar profesional. Te recomendamos usarlo siempre.

---

## 3. Histograma — ¿cómo se distribuye una variable?

Un **histograma** divide el rango de una variable numérica en intervalos (`bins`) y cuenta cuántos valores caen en cada uno. Sirve para ver:

- La **forma** de la distribución (simétrica, sesgada, bimodal...).
- **Outliers** (valores atípicos en los extremos).
- El **valor más común** (la moda).

### Demo — distribución de `age`

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df_clean["age"], bins=20, edgecolor="black")
ax.set_title("Distribución de edad de los pasajeros")
ax.set_xlabel("Edad")
ax.set_ylabel("Número de pasajeros")
plt.show()

👀 Observa el pico alrededor de los 28 años — ese es el efecto de haber **imputado con la mediana** en el Notebook 06. Antes había 177 NaN; ahora todos tienen 28.

### 🏋️ Ejercicio 1 — Histograma de `fare`

Crea un histograma de la columna **`fare`** (precio del billete) de `df_clean`.

Tu código debe:

1. Crear la figura con `fig1, ax1 = plt.subplots(figsize=(8, 5))`.
2. Dibujar el histograma con **30 bins** sobre `ax1`.
3. Poner el **título** `"Distribución del precio del billete"`.
4. Poner la **etiqueta del eje X** como `"Fare"`.
5. Poner la **etiqueta del eje Y** como `"Frecuencia"`.

💡 Tip: usa `ax1.hist(...)`, `ax1.set_title(...)`, `ax1.set_xlabel(...)`, `ax1.set_ylabel(...)`.

In [ ]:
# YOUR CODE HERE
fig1, ax1 = plt.subplots(figsize=(8, 5))

# ... draw the histogram, set title, labels ...

plt.show()

In [ ]:
# Tests
assert ax1.get_title() == "Distribución del precio del billete", \
    f"Title must be 'Distribución del precio del billete', got '{ax1.get_title()}'"
assert ax1.get_xlabel() == "Fare", f"X label must be 'Fare', got '{ax1.get_xlabel()}'"
assert ax1.get_ylabel() == "Frecuencia", f"Y label must be 'Frecuencia', got '{ax1.get_ylabel()}'"
assert len(ax1.patches) == 30, f"Expected 30 bins (patches), got {len(ax1.patches)}"

# The total area should match the number of fare observations
total_count = sum(p.get_height() for p in ax1.patches)
assert int(total_count) == len(df_clean), \
    f"Histogram should cover all {len(df_clean)} fare values, got {int(total_count)}"

print("✅ ¡Bien! Tu primer histograma con título y etiquetas correctas.")

---

## 4. Gráfico de barras — comparar categorías

Un **gráfico de barras** muestra un valor (altura) por cada categoría (eje X). Perfecto para comparar conteos o promedios entre grupos.

Hay dos formas comunes de hacerlo:

```python
# Opción A — directamente con matplotlib
ax.bar(["A", "B", "C"], [10, 20, 15])

# Opción B — desde un Series de pandas (más cómodo si ya hiciste un groupby)
my_series.plot.bar(ax=ax)
```

Vamos con la opción B porque encaja perfectamente con `value_counts()` y `groupby()` que ya conoces del Notebook 04.

### Demo — pasajeros por puerto de embarque

In [ ]:
# Compute the data first (NB04 territory)
embarked_counts = df_clean["embarked"].value_counts()
print(embarked_counts)

# Then plot
fig, ax = plt.subplots(figsize=(7, 4))
embarked_counts.plot.bar(ax=ax, color="steelblue", edgecolor="black")
ax.set_title("Pasajeros por puerto de embarque")
ax.set_xlabel("Puerto")
ax.set_ylabel("Número de pasajeros")
plt.xticks(rotation=0)   # avoid tilted labels
plt.show()

### 🏋️ Ejercicio 2 — Pasajeros por clase

Combina lo aprendido en NB04 con matplotlib.

1. Crea una `Series` llamada **`class_counts`** con el número de pasajeros **por clase** (columna `class`), usando `value_counts()`.
2. Crea `fig2, ax2 = plt.subplots(figsize=(7, 4))`.
3. Dibuja un gráfico de barras con `class_counts.plot.bar(ax=ax2)`.
4. Pon el título `"Pasajeros por clase"`.
5. Pon la etiqueta del eje Y como `"Número de pasajeros"`.

In [ ]:
# YOUR CODE HERE
class_counts = None

fig2, ax2 = plt.subplots(figsize=(7, 4))

# ... plot, title, ylabel ...

plt.show()

In [ ]:
# Tests
assert isinstance(class_counts, pd.Series), "class_counts must be a pandas Series"
assert len(class_counts) == 3, f"Expected 3 classes, got {len(class_counts)}"
assert set(class_counts.index) == {"First", "Second", "Third"}, "Class names mismatch"
assert class_counts["Third"] == 491, f"Third class should have 491 passengers, got {class_counts['Third']}"

# Plot checks
assert ax2.get_title() == "Pasajeros por clase", \
    f"Title must be 'Pasajeros por clase', got '{ax2.get_title()}'"
assert ax2.get_ylabel() == "Número de pasajeros", \
    f"Y label must be 'Número de pasajeros', got '{ax2.get_ylabel()}'"
assert len(ax2.patches) == 3, f"Expected 3 bars, got {len(ax2.patches)}"

print("✅ ¡Genial! Combinaste value_counts() con un gráfico de barras.")
print(class_counts)

---

## 5. Barras con `groupby` — comparar promedios

El paso siguiente: en vez de **contar** filas por categoría, **calculas un promedio por categoría** (lo que hiciste en NB04 con `groupby`) y lo grafica.

### Demo — tasa de supervivencia por clase

In [ ]:
# Compute survival rate per class (NB04 + NB06)
survival_by_class = df_clean.groupby("class", observed=True)["survived"].mean()
print(survival_by_class)

fig, ax = plt.subplots(figsize=(7, 4))
survival_by_class.plot.bar(ax=ax, color=["#2ecc71", "#f39c12", "#e74c3c"], edgecolor="black")
ax.set_title("Tasa de supervivencia por clase")
ax.set_xlabel("Clase")
ax.set_ylabel("Tasa de supervivencia")
ax.set_ylim(0, 1)   # survival rate is between 0 and 1
plt.xticks(rotation=0)
plt.show()

### 🏋️ Ejercicio 3 — Tasa de supervivencia por sexo

1. Crea una `Series` llamada **`survival_by_sex`** con la tasa de supervivencia (`survived` promedio) **por sexo**.
2. Crea `fig3, ax3 = plt.subplots(figsize=(6, 4))`.
3. Dibuja un gráfico de barras con `survival_by_sex.plot.bar(ax=ax3)`.
4. Pon el título `"Tasa de supervivencia por sexo"`.
5. Pon la etiqueta del eje Y como `"Tasa de supervivencia"`.

In [ ]:
# YOUR CODE HERE
survival_by_sex = None

fig3, ax3 = plt.subplots(figsize=(6, 4))

# ... plot, title, ylabel ...

plt.show()

In [ ]:
# Tests
assert isinstance(survival_by_sex, pd.Series), "survival_by_sex must be a pandas Series"
assert len(survival_by_sex) == 2, f"Expected 2 sexes, got {len(survival_by_sex)}"
assert set(survival_by_sex.index) == {"female", "male"}, "Index must be {'female', 'male'}"
assert np.isclose(survival_by_sex["female"], 0.7404, atol=0.005), \
    f"Female survival rate should be ~0.74, got {survival_by_sex['female']:.3f}"
assert np.isclose(survival_by_sex["male"], 0.1889, atol=0.005), \
    f"Male survival rate should be ~0.19, got {survival_by_sex['male']:.3f}"

# Plot checks
assert ax3.get_title() == "Tasa de supervivencia por sexo", \
    f"Title mismatch, got '{ax3.get_title()}'"
assert ax3.get_ylabel() == "Tasa de supervivencia", \
    f"Y label mismatch, got '{ax3.get_ylabel()}'"
assert len(ax3.patches) == 2, f"Expected 2 bars, got {len(ax3.patches)}"

print("✅ ¡Bien! Visualizaste la diferencia más impactante del dataset.")
print(survival_by_sex)

---

## 6. Scatter plot — relación entre dos variables numéricas

Un **scatter plot** dibuja un punto por cada fila del DataFrame, usando dos columnas como coordenadas (X, Y). Sirve para ver **si hay relación** entre dos variables numéricas.

```python
ax.scatter(x_data, y_data, alpha=0.5)
```

`alpha=0.5` hace los puntos semitransparentes — muy útil cuando hay solapamiento.

### Demo — `age` vs `fare`

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df_clean["age"], df_clean["fare"], alpha=0.4)
ax.set_title("Edad vs precio del billete")
ax.set_xlabel("Edad")
ax.set_ylabel("Fare")
plt.show()

👀 Observa que **no hay una relación lineal clara**: pasajeros de todas las edades pagaron precios muy variables. La columna `fare` tiene varios outliers extremos (>500).

### 🏋️ Ejercicio 4 — Scatter `sibsp` vs `fare`

Vamos a ver si hay relación entre el **número de hermanos/cónyuges a bordo** (`sibsp`) y el **precio del billete** (`fare`).

1. Crea `fig4, ax4 = plt.subplots(figsize=(7, 5))`.
2. Dibuja un scatter de `sibsp` (eje X) contra `fare` (eje Y) con `alpha=0.4`.
3. Pon el título `"Hermanos a bordo vs precio del billete"`.
4. Pon `"Número de hermanos / cónyuges (sibsp)"` como etiqueta del eje X.
5. Pon `"Fare"` como etiqueta del eje Y.

In [ ]:
# YOUR CODE HERE
fig4, ax4 = plt.subplots(figsize=(7, 5))

# ... scatter, title, labels ...

plt.show()

In [ ]:
# Tests
assert ax4.get_title() == "Hermanos a bordo vs precio del billete", \
    f"Title mismatch, got '{ax4.get_title()}'"
assert ax4.get_xlabel() == "Número de hermanos / cónyuges (sibsp)", \
    f"X label mismatch, got '{ax4.get_xlabel()}'"
assert ax4.get_ylabel() == "Fare", f"Y label mismatch, got '{ax4.get_ylabel()}'"

# A scatter plot stores its points in a 'collection'
assert len(ax4.collections) >= 1, "ax4 should contain a scatter collection"
n_points = len(ax4.collections[0].get_offsets())
assert n_points == len(df_clean), \
    f"Expected {len(df_clean)} points (one per passenger), got {n_points}"

print("✅ ¡Excelente! Hiciste tu primer scatter plot.")

---

## 7. Boxplot — comparar distribuciones por categoría

Un **boxplot** (gráfico de caja) resume una distribución mostrando:

- La **mediana** (línea dentro de la caja).
- El **rango intercuartílico** (caja: del Q1 al Q3, contiene el 50% central).
- Los **bigotes** (extremos de la distribución sin contar outliers).
- Los **outliers** (puntos fuera de los bigotes).

Es perfecto para **comparar distribuciones** entre varias categorías de un solo vistazo.

```python
data = [series_grupo1, series_grupo2, series_grupo3]
ax.boxplot(data, tick_labels=["Grupo 1", "Grupo 2", "Grupo 3"])
```

### Demo — distribución de `age` por sexo

In [ ]:
ages_female = df_clean[df_clean["sex"] == "female"]["age"]
ages_male = df_clean[df_clean["sex"] == "male"]["age"]

fig, ax = plt.subplots(figsize=(7, 5))
ax.boxplot([ages_female, ages_male], tick_labels=["female", "male"])
ax.set_title("Distribución de edad por sexo")
ax.set_ylabel("Edad")
plt.show()

### 🏋️ Ejercicio 5 — Boxplot `fare` por clase

Compara la distribución de **`fare`** entre las 3 clases de billete.

1. Crea tres `Series` (una por clase): **`fare_first`**, **`fare_second`**, **`fare_third`** filtrando `df_clean` por `class`.
2. Crea `fig5, ax5 = plt.subplots(figsize=(8, 5))`.
3. Dibuja un boxplot con `ax5.boxplot([fare_first, fare_second, fare_third], tick_labels=["First", "Second", "Third"])`.
4. Pon el título `"Distribución del precio por clase"`.
5. Pon `"Fare"` como etiqueta del eje Y.

In [ ]:
# YOUR CODE HERE
fare_first = None
fare_second = None
fare_third = None

fig5, ax5 = plt.subplots(figsize=(8, 5))

# ... boxplot, title, ylabel ...

plt.show()

In [ ]:
# Tests
assert isinstance(fare_first, pd.Series), "fare_first must be a pandas Series"
assert isinstance(fare_second, pd.Series), "fare_second must be a pandas Series"
assert isinstance(fare_third, pd.Series), "fare_third must be a pandas Series"
assert len(fare_first) == 214, f"First class should have 214 passengers, got {len(fare_first)}"
assert len(fare_second) == 184, f"Second class should have 184 passengers, got {len(fare_second)}"
assert len(fare_third) == 491, f"Third class should have 491 passengers, got {len(fare_third)}"

# 1st-class fares should be higher than 3rd-class on average
assert fare_first.mean() > fare_third.mean(), \
    "First-class fares should be higher than third-class on average"

# Plot checks
assert ax5.get_title() == "Distribución del precio por clase", \
    f"Title mismatch, got '{ax5.get_title()}'"
assert ax5.get_ylabel() == "Fare", f"Y label mismatch, got '{ax5.get_ylabel()}'"

xtick_labels = [t.get_text() for t in ax5.get_xticklabels()]
assert xtick_labels == ["First", "Second", "Third"], \
    f"X tick labels must be ['First','Second','Third'], got {xtick_labels}"

print("✅ ¡Genial! Tu boxplot revela algo importante:")
print(f"   1st class fare median: {fare_first.median():.2f}")
print(f"   2nd class fare median: {fare_second.median():.2f}")
print(f"   3rd class fare median: {fare_third.median():.2f}")

---

## 8. Resumen — ¿qué aprendiste?

🎉 ¡Ahora puedes contar historias con datos!

| Tipo de gráfico | Cuándo usarlo | Sintaxis principal |
|---|---|---|
| Histograma | Distribución de **una** variable numérica | `ax.hist(s, bins=N)` |
| Barras | Comparar valores entre **categorías** | `series.plot.bar(ax=ax)` |
| Scatter | Relación entre **dos** variables numéricas | `ax.scatter(x, y, alpha=0.5)` |
| Boxplot | Comparar **distribuciones** entre categorías | `ax.boxplot([s1, s2, s3], tick_labels=[...])` |

### Reglas prácticas

1. **Siempre** pon **título** y **etiquetas** en los ejes — sin eso, un gráfico no comunica nada.
2. Usa `figsize=(ancho, alto)` para ajustar el tamaño cuando los datos no caben.
3. Para muchos puntos solapados en un scatter, usa `alpha < 1` (semitransparencia).
4. Cuando vayas a graficar resultados de un `groupby` o `value_counts`, **calcula primero la `Series`** y luego haz `series.plot.bar(ax=ax)`.

## ¿Qué viene en el próximo notebook?

En **Notebook 08 — Tu primer modelo de Machine Learning** vas a entrenar una **regresión lineal** con `scikit-learn` que aprenda a predecir el precio del billete (`fare`) a partir de otras columnas. Es el primer paso del ML supervisado. ¡Nos vemos ahí! 🤖